In [ ]:
#setup
from google.colab import drive
drive.mount("/content/drive")
import subprocess, os, sys, json
CODE_DIR = "/content/ecEvol"
REPO_URL = "https://github.com/bdamerac/ecEvol.git"
subprocess.run(["git","clone",REPO_URL,CODE_DIR] if not os.path.exists(CODE_DIR) else ["git","-C",CODE_DIR,"pull"], check=True)
subprocess.run(["pip","install","-q","h5py"], check=True)
sys.path.insert(0, os.path.join(CODE_DIR,"scripts"))

import numpy as np, pandas as pd, torch
from train import TrainConfig, train
from artifacts import save_preds, save_table


In [ ]:
BASE_DIR = "/content/drive/MyDrive/evolution/f24"          # inputs (sim H5, pcawg/, hmf/)
OUT_ROOT = "/content/drive/MyDrive/ecdna_evolution_runs"   # all outputs
H5_PATH    = os.path.join(BASE_DIR, "all_simulations_20k.h5")
DATA_DIR   = os.path.join(OUT_ROOT, "splits")
MODELS_DIR = os.path.join(OUT_ROOT, "checkpoints")
RUNS_DIR   = os.path.join(OUT_ROOT, "runs")
LOGS_DIR   = os.path.join(OUT_ROOT, "logs")
PRED_DIR   = os.path.join(OUT_ROOT, "predictions")
RES_DIR    = os.path.join(LOGS_DIR, "results")
for d in (DATA_DIR, MODELS_DIR, RUNS_DIR, LOGS_DIR, PRED_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)

DEPTHS, SEED = [30, 50, 100], 42
ABLATION_ARMS = [
    ("cnn_attn",        dict(backbone="cnn_attn")),
    ("cnn_attn_pool",   dict(backbone="cnn_attn_pool")),
    ("cnn_noattn",      dict(backbone="cnn_noattn")),
    ("cnn_2ch",         dict(backbone="cnn_2ch")),
    ("r1_only",         dict(backbone="r1_only")),
    ("noaug",           dict(backbone="cnn_attn", augment=False)),
    ("singletask_s",    dict(backbone="cnn_attn", heads=("s",))),
]
HP_GRID = [(lr, wd, bs) for lr in (5e-4, 1e-3) for wd in (0.0, 1e-4, 1e-3) for bs in (64, 128)]

# cohort inputs (edit to your Drive paths)
PCAWG_AFS_PKL  = os.path.join(BASE_DIR, "pcawg", "icgc_tcga_amplicon_vcfs_ecDNA_outside.pkl")
PCAWG_META_TSV = os.path.join(BASE_DIR, "pcawg", "pcawg_feature_metadata.tsv")
HMF_DIR        = "/content/drive/MyDrive/evolution/HMF_processed_data"  # hmf inputs live outside BASE_DIR
HMF_CACHE_CSV  = os.path.join(HMF_DIR, "hmf_ecdna_af_cache_ss_ms.csv")
HMF_META_TSV   = os.path.join(HMF_DIR, "hmf_ecdna_feature_metadata_merged.tsv")
COVERAGE_COL, CN_COL = "depth", "median_feature_cn"
PHI_CKPTS = {d: os.path.join(MODELS_DIR, f"phi_depth{d}.pt") for d in DEPTHS}  # trained separately

def train_cached(cfg, ckpt_name, run_dir=None):
    """Train unless a finished result for this checkpoint already exists (resumable)."""
    res_path = os.path.join(RES_DIR, ckpt_name.replace(".pt", ".json"))
    if os.path.exists(os.path.join(MODELS_DIR, ckpt_name)) and os.path.exists(res_path):
        print(f"[skip] {ckpt_name}"); return json.load(open(res_path))
    res = train(cfg, H5_PATH, DATA_DIR, MODELS_DIR, ckpt_name, device=DEVICE,
                run_dir=run_dir, save_preds_fn=save_preds, verbose=True)
    json.dump(res, open(res_path, "w"), indent=2, default=float)
    return res

In [ ]:
#ablations
abl = []
for label, ov in ABLATION_ARMS:
    cfg = TrainConfig(depth=100, **ov)
    r = train_cached(cfg, f"abl_{label}.pt", run_dir=os.path.join(RUNS_DIR, "ablations"))
    abl.append({"arm": label, **r})
abl = pd.DataFrame(abl)
S5 = abl[["arm","test_spear_s","test_rmse_mu","test_r2_mu","test_mae_g",
          "test_acc_exact_g","test_acc_pm1_g","testpert_spear_s"]].round(3)
save_table(S5, os.path.join(LOGS_DIR, "ablations.csv")); display(S5)

# The two cross-attention variants are statistically indistinguishable on the
# validation composite; pick the parameter-efficient one.
cand = abl[abl["arm"].isin(["cnn_attn", "cnn_attn_pool"])].copy()
near = cand[cand["best_val_score"] >= cand["best_val_score"].max() - 0.05]
ARCH = near.sort_values("n_params").iloc[0]["backbone"]
display(cand[["arm", "best_val_score", "n_params", "test_spear_s", "testpert_spear_s"]].round(3))
print("chosen architecture:", ARCH)

In [ ]:
#HP tuning on the chosen architecture (depth 100)
hp = []
for lr, wd, bs in HP_GRID:
    cfg = TrainConfig(backbone=ARCH, depth=100, lr=lr, weight_decay=wd, batch_size=bs)
    r = train_cached(cfg, f"hp_{ARCH}_lr{lr:g}_wd{wd:g}_bs{bs}.pt")
    hp.append({"lr": lr, "wd": wd, "bs": bs, "val_composite": r["best_val_score"],
               "test_spear_s": r["test_spear_s"], "test_r2_s": r["test_r2_s"],
               "testpert_spear_s": r["testpert_spear_s"]})
hp = pd.DataFrame(hp).sort_values("val_composite", ascending=False).reset_index(drop=True)
save_table(hp, os.path.join(LOGS_DIR, "hp_tuning.csv")); display(hp)

best = hp.iloc[0]
within = hp[hp["val_composite"] >= best["val_composite"] - hp["val_composite"].std()]
dflt = within[(within.lr == 1e-3) & (within.wd == 1e-4) & (within.bs == 128)]
row = dflt.iloc[0] if len(dflt) else best
WIN = {"backbone": ARCH, "lr": float(row["lr"]), "weight_decay": float(row["wd"]), "batch_size": int(row["bs"])}
print("selected config:", WIN)

In [ ]:
#phi (descendant-fraction) models, one per depth
from phi_model import train_phi
phi_res = []
for d in DEPTHS:
    ck = f"phi_depth{d}.pt"
    rp = os.path.join(RES_DIR, ck.replace(".pt", ".json"))
    if os.path.exists(os.path.join(MODELS_DIR, ck)) and os.path.exists(rp):
        print(f"[skip] {ck}"); phi_res.append(json.load(open(rp))); continue
    r = train_phi(d, H5_PATH, DATA_DIR, MODELS_DIR, ck, device=DEVICE, verbose=True)
    json.dump(r, open(rp, "w"), indent=2, default=float); phi_res.append(r)
phi_df = pd.DataFrame(phi_res)[["depth", "test_r2", "test_mae", "test_pearson"]].round(3)
save_table(phi_df, os.path.join(LOGS_DIR, "phi_metrics_by_depth.csv")); display(phi_df)

In [ ]:
#final per-depth models
from train import get_or_create_splits
from dataset import ECDNADataset
from infer import predict_phi_on_dataset
RUN_FINAL = os.path.join(RUNS_DIR, "final")
res = []
for depth in DEPTHS:
    cfg = TrainConfig(depth=depth, **WIN)
    r = train_cached(cfg, f"final_{ARCH}_d{depth}_s{SEED}.pt", run_dir=RUN_FINAL); r["depth"]=depth
    res.append(r)
    if os.path.exists(PHI_CKPTS[depth]):
        ds = ECDNADataset(H5_PATH, depth=depth, augment=False)
        _, _, test_idx, _ = get_or_create_splits(cfg, ds, DATA_DIR)
        save_preds(predict_phi_on_dataset(PHI_CKPTS[depth], ds, test_idx, device=DEVICE),
                   os.path.join(RUN_FINAL, "preds", f"phi_sim_depth{depth}.parquet"))
res = pd.DataFrame(res)

S1 = pd.DataFrame([{"parameter": p, "depth": int(r["depth"]), "R2": round(r[f"test_r2_{k}"],3), "MAE": round(r[f"test_mae_{k}"],3)}
                   for _, r in res.iterrows() for p, k in [("gintro","g"),("mu","mu"),("s","s")]])
S2 = res[["depth","test_ppv_s_high","test_ppv_s_low"]].round(3)
S3 = res[["depth","test_round_exact_g","test_round_pm1_g"]].round(3)
for n, t in [("sim_metrics_by_depth",S1),("selection_ppv_by_depth",S2),("emergence_rounding_by_depth",S3)]:
    save_table(t, os.path.join(LOGS_DIR, f"{n}.csv"))
display(S1); display(S2); display(S3)

In [ ]:
#ABC baseline
import abc_baseline as ABC
s4_path = os.path.join(LOGS_DIR, "dl_vs_abc.csv")
if os.path.exists(s4_path):
    display(pd.read_csv(s4_path))
else:
    abc_clean = ABC.run_abc(H5_PATH, DATA_DIR, depth=100, augmented=False, max_ref=200000)
    abc_aug   = ABC.run_abc(H5_PATH, DATA_DIR, depth=100, augmented=True,  max_ref=200000)
    dl_aug   = train_cached(TrainConfig(backbone=ARCH, depth=100, augment=True),  f"s4_dl_aug_{ARCH}.pt")
    dl_clean = train_cached(TrainConfig(backbone=ARCH, depth=100, augment=False), f"s4_dl_clean_{ARCH}.pt")
    def dl_rows(name, r):
        return [{"method":name,"test_set":t,"rho_s":r[f"{p}spear_s"],"rmse_s":r[f"{p}rmse_s"],
                 "r2_mu":r[f"{p}r2_mu"],"rmse_mu":r[f"{p}rmse_mu"],
                 "exact_g":r[f"{p}acc_exact_g"],"pm1_g":r[f"{p}acc_pm1_g"]}
                for t,p in [("clean","test_"),("perturbed","testpert_")]]
    def abc_rows(name, res):
        return [{"method":name,"test_set":t,"rho_s":m["spear_s"],"rmse_s":m["rmse_s"],"r2_mu":m["r2_mu"],
                 "rmse_mu":m["rmse_mu"],"exact_g":m["acc_exact_g"],"pm1_g":m["acc_pm1_g"]}
                for t,m in [("clean",res["metrics_clean"]),("perturbed",res["metrics_perturbed"])]]
    S4 = pd.DataFrame(dl_rows("model, purity-aug", dl_aug)+dl_rows("model, clean", dl_clean)
                      +abc_rows("ABC, clean ref", abc_clean)+abc_rows("ABC, aug ref", abc_aug)).round(3)
    save_table(S4, s4_path); display(S4)

In [ ]:
#cohort inference: ensemble across seeds
#trains the final config at each ensemble seed, scores both cohorts per seed
#({cohort}_pred_{ARCH}_s{seed}.tsv), and builds the averaged ensemble.
#pcawg is depth-matched; hmf (~100x) is scored with the depth-100 model.
from infer import (load_pcawg_afs, make_afs_loader, make_hmf_loader,
                   run_inference, run_inference_depth_matched, ensemble_predictions)
SEEDS_ENS = list(range(42, 52))                                   # 10-network ensemble
DEPLOY_DEPTH = 100                                                # hmf coverage ~ 100x
PHI_OK = {d: p for d, p in PHI_CKPTS.items() if os.path.exists(p)}

# cohort -> (metadata, loader, mode, cn_col, id_col)
cohorts = {
    "pcawg": (PCAWG_META_TSV, make_afs_loader(load_pcawg_afs(PCAWG_AFS_PKL), min_mutations=10),
              "depth_matched", CN_COL, "Feature_ID"),
    "hmf":   (HMF_META_TSV, make_hmf_loader(HMF_CACHE_CSV, min_mutations=10),
              "fixed_depth", "copy_number_median", "feature_id_full"),
}

seed_paths = {c: [] for c in cohorts}
for sd in SEEDS_ENS:
    for depth in DEPTHS:
        train_cached(TrainConfig(depth=depth, seed=sd, **WIN), f"final_{ARCH}_d{depth}_s{sd}.pt", run_dir=RUN_FINAL)
    ck = {d: os.path.join(MODELS_DIR, f"final_{ARCH}_d{d}_s{sd}.pt") for d in DEPTHS}
    for cohort, (meta_tsv, loader, mode, cn_col, id_col) in cohorts.items():
        pp = os.path.join(PRED_DIR, f"{cohort}_pred_{ARCH}_s{sd}.tsv")
        if not os.path.exists(pp):
            if mode == "depth_matched":
                run_inference_depth_matched(meta_tsv, loader, ck, PHI_OK, pp, depths=DEPTHS,
                    coverage_col=COVERAGE_COL, cn_col=cn_col, device=DEVICE)
            else:
                run_inference(meta_tsv, loader, ck[DEPLOY_DEPTH], PHI_OK.get(DEPLOY_DEPTH), pp,
                    cn_col=cn_col, device=DEVICE)
        seed_paths[cohort].append(pp)

for cohort, (*_, id_col) in cohorts.items():                      # averaged ensemble per cohort
    ensemble_predictions(seed_paths[cohort],
                         os.path.join(PRED_DIR, f"{cohort}_pred_{ARCH}_ensemble.tsv"), id_col=id_col)

In [ ]:
#population-size generalisation
#the same simulated trajectory is checkpointed at 10k..50k cells and scored at each
#checkpoint, so predictions are compared within a simulation against its own 20k value.
#requires checkpointed_simulations_merged_so_far.h5 in BASE_DIR.
import h5py
from infer import load_evo_model, load_phi_model, _predict_batch

POPSIZE_H5 = os.path.join(BASE_DIR, "checkpointed_simulations_merged_so_far.h5")
SIZES, POP_DEPTH = [10000, 20000, 30000, 40000, 50000], 100
MIN_DESC = 4000                                   # retention rule from the original design

rows = []
for seed in SEEDS_ENS:
    evo_ck = os.path.join(MODELS_DIR, f"final_{ARCH}_d{POP_DEPTH}_s{seed}.pt")
    if not os.path.exists(evo_ck):
        print("missing checkpoint, skipping:", os.path.basename(evo_ck)); continue
    evo, mu_scaler = load_evo_model(evo_ck, DEVICE)
    phi_ck = PHI_CKPTS[POP_DEPTH]
    phi = load_phi_model(phi_ck, DEVICE) if os.path.exists(phi_ck) else None

    with h5py.File(POPSIZE_H5, "r") as f:
        for sim in f:
            g = f[sim]
            if not all(f"pop_{n}" in g for n in SIZES):
                continue
            if any(g[f"pop_{n}"].attrs["tot_descended"] < MIN_DESC for n in SIZES):
                continue
            r1_b, r2_b, cn_b, meta = [], [], [], []
            for n in SIZES:
                ck = g[f"pop_{n}"]
                r1_b.append(ck[f"r1_spectrum_{POP_DEPTH}"][:].astype(np.float32))
                r2_b.append(ck[f"r2_spectrum_{POP_DEPTH}"][:].astype(np.float32))
                cn_b.append(float(ck.attrs["ecdna_region_cn"]))
                meta.append((n, float(ck.attrs["tot_descended"]) / n))
            phi_h, s_h, mu_h, g_exp, g_arg, _ = _predict_batch(
                evo, phi, mu_scaler, r1_b, r2_b, cn_b, DEVICE)
            for i, (n, phi_true) in enumerate(meta):
                rows.append(dict(seed=seed, simulation_id=int(g.attrs["simulation_id"]),
                                 population_size=n, phi_true=phi_true,
                                 true_s=float(g.attrs["ecdna_s"]),
                                 true_intro=int(g.attrs["intro_generation"]),
                                 true_mu=float(g.attrs["mu"]),
                                 phi_hat=phi_h[i], s_hat=s_h[i], mu_hat=mu_h[i],
                                 g_exp=g_exp[i], g_argmax=int(g_arg[i])))

pop = pd.DataFrame(rows)
ens_pop = (pop.drop(columns=["seed"])
              .groupby(["simulation_id", "population_size"]).mean(numeric_only=True)
              .reset_index())
ref = ens_pop[ens_pop.population_size == 20000].set_index("simulation_id")
for c in ["phi_hat", "s_hat", "mu_hat", "g_exp"]:
    ens_pop[f"delta_{c}_vs_20k"] = ens_pop[c].values - ens_pop.simulation_id.map(ref[c]).values

out = os.path.join(PRED_DIR, f"popsize_pred_{ARCH}_d{POP_DEPTH}_ensemble.tsv")
ens_pop.to_csv(out, sep="\t", index=False)
print(f"wrote {out}  ({ens_pop.simulation_id.nunique()} simulations x {len(SIZES)} sizes, "
      f"{pop.seed.nunique()} seeds)")
display(ens_pop.groupby("population_size")[
    ["delta_s_hat_vs_20k", "delta_mu_hat_vs_20k", "delta_g_exp_vs_20k"]].median().round(3))
